## Carregando os dados

In [1]:
import pandas as pd 
import os

BASE_DIR = '../data'
TRAIN_CSV_PATH = os.path.join(BASE_DIR, 'train.csv')

df_train = pd.read_csv(TRAIN_CSV_PATH)

In [2]:
print(f"Tamanho do dataset de treino: {df_train.shape}")

Tamanho do dataset de treino: (106800, 15)


In [3]:
display(df_train.head())

,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0


## Entendendo os dados

**1. Onde buscar os dados (IDs)**
* **`eeg_id` (1628180742):** Indica que você deve abrir o arquivo `1628180742.parquet` na pasta de EEGs.
* **`spectrogram_id` (353733):** Indica que você deve abrir o arquivo `353733.parquet` na pasta de espectrogramas.
* **`patient_id` (42516):** Identificador único do paciente.

**2. Offsets**
Os arquivos `.parquet` contêm minutos ou até horas de gravação, mas o modelo só vai olhar para trechos de 50 segundos.
* **`eeg_label_offset_seconds` (0.0):** Diz para você: "Abra o arquivo de EEG e comece a ler a partir do segundo 0". Se fosse `28.0`, você teria que ignorar os primeiros 28 segundos do arquivo e fatiar os 50 segundos logo depois disso.
* **`spectrogram_label_offset_seconds` (0.0):** A mesma lógica, mas para o arquivo de espectrograma e fatiando em 10 minutos. 

**3. O Diagnóstico (A Variável Alvo/Target)**
* **`expert_consensus` (Seizure):** A palavra final. A maioria dos médicos concordou que esse recorte é uma Convulsão (*Seizure*).
* **Os Votos (`seizure_vote` 3, `lpd_vote` 0, etc.):** Isso mostra que 3 médicos votaram que era Convulsão, e nenhum médico votou nas outras anomalias. 


### Objetivo:

A partir dos 50 segundos descritos na tabela do eletrocardiograma e do espectograma, nosso modelo, a partir de uma distribuição de probabilidade para a atividade cerebral prejudicial daquele paciente. 

Como os dados vêm de pacientes que já estão internados em estado crítico (UTI), é raro ter um cérebro "100% perfeitamente normal". No entanto, quando os neurologistas olham para a tela e não identificam nenhuma das 5 anomalias graves (Seizure, LPD, GPD, LRDA, GRDA), eles votam em "Other". Assim, "Other" representa:

* Atividade de base normal (para aquele paciente).

* Artefatos: O sinal elétrico do EEG é muito sensível. Se o paciente pisca os olhos, mastiga, ou se alguém esbarra no fio do eletrodo na UTI, isso gera um pico gigantesco no gráfico que parece uma anomalia, mas é só ruído.

* Lentidão inespecífica: O cérebro está um pouco lento por causa de remédios sedativos, mas não há danos agudos.

## Preparação dos dados


O modelo vai olhar para seizure_vote para definir uma distribuição de probabilidade para "seizure", de forma que se um caso teve 3 votos para Seizure e nenhum para o resto, a probabilidade é 100% (1.0). Se houve divergência, o modelo precisa aprender essa incerteza.

In [4]:
target_cols = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

# Somamos os votos totais para cada linha
y_data = df_train[target_cols].values
y_data_sum = y_data.sum(axis=1, keepdims=True)

# Transformamos em probabilidade (0 a 1)
y_data = y_data / y_data_sum

# Atualizamos o dataframe com as probabilidades
df_train[target_cols] = y_data

In [5]:
df_train.head()

,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,1.0,0.0,0.0,0.0,0.0,0.0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,1.0,0.0,0.0,0.0,0.0,0.0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,1.0,0.0,0.0,0.0,0.0,0.0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,1.0,0.0,0.0,0.0,0.0,0.0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,1.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# Agrupando pelas 3 chaves principais e contando quantas linhas (medições) existem
analise_exames = df_train.groupby(
    ['patient_id', 'eeg_id', 'spectrogram_id']
).size().reset_index(name='qtd_medicoes')

# Ordenando do exame com mais medições para o com menos
analise_exames = analise_exames.sort_values('qtd_medicoes', ascending=False)

print(f"Total de exames únicos: {len(analise_exames)}")
print("\nTop 10 exames com mais recortes (medições):")
display(analise_exames.head(10))

Total de exames únicos: 17089

Top 10 exames com mais recortes (medições):


,patient_id,eeg_id,spectrogram_id,qtd_medicoes
672,2641,2259539799,1391458063,743
15159,57378,2428433259,1974785580,664
2422,10187,1641054670,577118473,562
689,2641,2860052642,840003147,534
6833,27761,525664301,365931891,531
13675,52100,1712056492,1443709916,433
1955,8033,1480985066,2063104016,416
6968,28330,188361788,1568768668,412
15113,56885,3525185677,2011177737,286
654,2641,1596590162,513069562,275


### Explicação 
Temos 106800 linhas para análise na nossa base de dados. Porém, elas representam apenas 17089 exames únicos, sendo recortes de tempo (50 segundos) diferentes desses mesmos 17089 exames. Para um MVP, podemos definir como objetivo utiliazar 1 recorte por exame, com as seguintes justificativas:

1. Caso os recortes se sobreponham, o modelo pode acabar decorando algum ruído de fundo (decora o paciente) e sofrer overfitting
2. Alguns exames têm centenas de medições e outros têm só uma. Pegando apenas a primeira de cada, garantimos que o modelo aprenda com 17089 exames diferentes, dando o mesmo peso para todos. 
3. Velocidade de iteração (treinar com 17 mil dados é mais rápido que treinar com 100 mil)


Assim sendo, vamos pegar a primeira aparição de cada exame-paciente (ou seja, a medição que começa no 0.0 seg, na maioria das vezes)

In [7]:
df_treino_mvp = df_train.groupby(
    ['patient_id', 'eeg_id', 'spectrogram_id']
).first().reset_index()

print(f"Tamanho do dataset do MVP: {df_treino_mvp.shape}")

Tamanho do dataset do MVP: (17089, 15)


In [8]:
# Filtrando o nosso MVP para achar onde a primeira medição NÃO é zero
exames_fora_do_zero = df_treino_mvp[df_treino_mvp['eeg_label_offset_seconds'] > 0.0]

quantidade = len(exames_fora_do_zero)
porcentagem = (quantidade / len(df_treino_mvp)) * 100

print(f"Total de exames que NÃO começam no 0.0: {quantidade}")
print(f"Isso representa {porcentagem:.2f}% do nosso MVP.\n")

if quantidade > 0:
    print("Exemplos de exames que começam 'atrasados':")
    display(exames_fora_do_zero[['patient_id', 'eeg_id', 'eeg_label_offset_seconds']].head())

Total de exames que NÃO começam no 0.0: 0
Isso representa 0.00% do nosso MVP.



In [9]:
# Dividindo grupos de colunas
colunas_id = ['eeg_id', 'spectrogram_id', 'patient_id', 'expert_consensus']
colunas_corte = ['eeg_label_offset_seconds', 'spectrogram_label_offset_seconds']
colunas_target = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

colunas_finais = colunas_id + colunas_corte + colunas_target
df_treino_mvp = df_treino_mvp[colunas_finais]

# Verificando valores nulos
print("Contagem de nulos por coluna antes da limpeza:")
print(df_treino_mvp.isnull().sum())

Contagem de nulos por coluna antes da limpeza:
eeg_id                              0
spectrogram_id                      0
patient_id                          0
expert_consensus                    0
eeg_label_offset_seconds            0
spectrogram_label_offset_seconds    0
seizure_vote                        0
lpd_vote                            0
gpd_vote                            0
lrda_vote                           0
grda_vote                           0
other_vote                          0
dtype: int64


Não temos valores nulos! Podemos seguir para a prepação final para partir para o treino do modelo

In [10]:
# Vamos salvar nossa base de treino
df_treino_mvp.to_csv('../data/train_mvp.csv', index=False)

## Criação de pastas com os arquivos transformados para .npy
Rodar apenas 1 vez

In [15]:
import numpy as np
from tqdm import tqdm
import os

# Criando a pasta para os eegs
PASTA_DESTINO = '../data/eeg_processado_mvp/'
os.makedirs(PASTA_DESTINO, exist_ok=True)

# Criando a pasta para os espectrogramas
PASTA_SPEC_DESTINO = '../data/spectrogram_processado_mvp/'
os.makedirs(PASTA_SPEC_DESTINO, exist_ok=True)

# A função de ligação
def processar_e_salvar_eeg(row):
    eeg_id = int(row['eeg_id'])
    offset = row['eeg_label_offset_seconds']
    
    # Abrir o parquet
    caminho_parquet = f"../data/train_eegs/{eeg_id}.parquet"
    df_eeg = pd.read_parquet(caminho_parquet)
    
    # Cortar os 50 segundos exatos
    start = int(offset * 200)
    end = start + 10000
    eeg_slice = df_eeg.iloc[start:end].values
    
    # Salvar a matriz individualmente
    caminho_salvar = os.path.join(PASTA_DESTINO, f"{eeg_id}.npy")
    np.save(caminho_salvar, eeg_slice)


def processar_e_salvar_spectrogram(row):
    spec_id = int(row['spectrogram_id'])
    offset = row['spectrogram_label_offset_seconds']
    
    # Abrir o parquet
    caminho_parquet = f"../data/train_spectrograms/{spec_id}.parquet"
    df_spec = pd.read_parquet(caminho_parquet)
    
    # A base do Kaggle salva os espectrogramas com linhas a cada 2 segundos.
    # Portanto, para achar a linha inicial, dividimos o offset por 2.
    # E pegamos as próximas 300 linhas (que equivalem a 600 segundos = 10 minutos)
    start = int(offset / 2)
    end = start + 300
    
    # Cortando a matriz (ignoramos a primeira coluna de 'time', pegando só as 400 colunas de frequência)
    # Se a coluna 'time' for a primeira (índice 0), pegamos da 1 em diante:
    spec_slice = df_spec.iloc[start:end, 1:].values
    
    # Salvar a matriz individualmente
    caminho_salvar = os.path.join(PASTA_SPEC_DESTINO, f"{spec_id}.npy")
    np.save(caminho_salvar, spec_slice)


In [18]:

# Rodando o processo para todo o MVP
if os.path.exists(PASTA_DESTINO):
    print(" Os arquivos de EEG já foram extraídos anteriormente. Pulando etapa...")
else:
    print("Iniciando extração das matrizes de EEG...")
    for index, row in tqdm(df_treino_mvp.iterrows(), total=len(df_treino_mvp)):
        try:
            processar_e_salvar_eeg(row)
        except Exception as e:
            print(f"Erro no eeg_id {row['eeg_id']}: {e}")

if os.path.exists(PASTA_SPEC_DESTINO):
    print(" Os arquivos de spectrogram já foram extraídos anteriormente. Pulando etapa...")
else:
    print("Iniciando extração das matrizes de Espectrograma...")
    for index, row in tqdm(df_treino_mvp.iterrows(), total=len(df_treino_mvp)):
        try:
            processar_e_salvar_spectrogram(row)
        except Exception as e:
            print(f"Erro no spectrogram_id {row['spectrogram_id']}: {e}")

 Os arquivos de EEG já foram extraídos anteriormente. Pulando etapa...
 Os arquivos de spectrogram já foram extraídos anteriormente. Pulando etapa...


In [20]:
# Dados de teste

import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# Carregando o mapa de teste
df_test = pd.read_csv('../data/test.csv')

# Criando as pastas de destino
PASTA_EEG_TESTE = '../data/eeg_teste_processado/'
PASTA_SPEC_TESTE = '../data/spectrogram_teste_processado/'
os.makedirs(PASTA_EEG_TESTE, exist_ok=True)
os.makedirs(PASTA_SPEC_TESTE, exist_ok=True)

if os.path.exists(PASTA_EEG_TESTE) and os.path.exists(PASTA_SPEC_TESTE):
    print(" Os arquivos de teste já foram extraídos anteriormente. Pulando etapa...")
else:
    print("VERIFICAÇÃO DE INTEGRIDADE DOS DADOS DE TESTE")

    # Pegando os IDs do primeiro exame da lista

    exemplo_eeg_id = int(df_test.loc[0, 'eeg_id'])
    exemplo_spec_id = int(df_test.loc[0, 'spectrogram_id'])

    # Lendo os parquets brutos
    exemplo_eeg_df = pd.read_parquet(f"../data/test_eegs/{exemplo_eeg_id}.parquet")
    exemplo_spec_df = pd.read_parquet(f"../data/test_spectrograms/{exemplo_spec_id}.parquet")

    print(f"Formato do EEG Bruto de Teste: {exemplo_eeg_df.shape}")
    print("Justificativa: O Kaggle já entrega exatas 10.000 linhas (50 segundos). Não há necessidade de usar Offsets (Tesoura)!\n")

    print(f"Formato do Espectrograma Bruto de Teste: {exemplo_spec_df.shape}")
    print("Justificativa: O Kaggle já entrega exatas 300 linhas (10 minutos). Faremos apenas a remoção da coluna 'time' e a conversão para Numpy.\n")

    print("CONCLUSÃO: Os dados de teste já vêm pré-recortados. Iniciando apenas a conversão para '.npy' para otimizar o tempo de inferência do modelo!")
    print("-" * 60 + "\n")

    # A Linha de montagem do EEG
    print("Iniciando extração do Teste (EEG)...")
    for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
        eeg_id = int(row['eeg_id'])
        
        # Lê o parquet e pega os valores diretamente (ele já tem o tamanho certo!)
        eeg_data = pd.read_parquet(f"../data/test_eegs/{eeg_id}.parquet").values
        np.save(os.path.join(PASTA_EEG_TESTE, f"{eeg_id}.npy"), eeg_data)

    # A Linha de montagem do Espectrograma
    print("Iniciando extração do Teste (Espectrograma)...")
    for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
        spec_id = int(row['spectrogram_id'])
        
        # Lê o parquet
        spec_data = pd.read_parquet(f"../data/test_spectrograms/{spec_id}.parquet")
        
        # Limpa a coluna de tempo (se ela veio no teste) e pega apenas as frequências
        if 'time' in spec_data.columns:
            spec_data = spec_data.drop(columns=['time'])
            
        np.save(os.path.join(PASTA_SPEC_TESTE, f"{spec_id}.npy"), spec_data.values)

    print("\n Dados de teste preparados e convertidos com sucesso!")

 Os arquivos de teste já foram extraídos anteriormente. Pulando etapa...


## Split e finalização do processo

In [12]:
from sklearn.model_selection import train_test_split

# Pegamos o nosso mapa do MVP que já estava limpo
df_mvp = pd.read_csv('../data/train_mvp.csv')

df_mvp.head()

,eeg_id,spectrogram_id,patient_id,expert_consensus,eeg_label_offset_seconds,spectrogram_label_offset_seconds,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,48274288,1496078390,56,Other,0.0,664.0,0.0,0.0,0.0,0.0,0.0,1.0
1,165634434,957002006,56,Other,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,201627859,688709208,56,Other,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,234570256,497667405,56,Other,0.0,110.0,0.0,0.0,0.0,0.0,0.0,1.0
4,271569269,925581305,56,Other,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [13]:
# Dividimos em Treino (80%) e Teste  (20%)
# O 'stratify' garante que a proporção de doenças seja a mesma nos dois grupos
# O 'random_state' garante que se você rodar de novo, a divisão será a mesma
df_treino_final, df_teste_final = train_test_split(
    df_mvp, 
    test_size=0.2, 
    random_state=42,
    stratify=df_mvp['expert_consensus'] # Se você manteve essa coluna
)

# Salvamos dois novos arquivos
df_treino_final.to_csv('../data/train_final.csv', index=False)
df_teste_final.to_csv('../data/test_final.csv', index=False)

print(f"Treino: {len(df_treino_final)} exames")
print(f"Teste (para métricas): {len(df_teste_final)} exames")

Treino: 13671 exames
Teste (para métricas): 3418 exames


A engenharia de dados inicial foi concluída (preparação dos dados). Abaixo está o guia dos próximos passos

#### Como estamos no momento:
Esqueça os arquivos .parquet gigantes e lentos. Os dados estão preparados para um processo mais rápido para o nosso MVP.

**Problema:** O kaggle, por padrão, disponibiliza 1 dado de teste para o teste final. Porém, como vamos medir a precisão do nosso modelo, vamos separar os dados que temos (que inicialmente seriam apenas treino) em treino e teste. Assim, a organização fica: 

1. **`train_mvp.csv`**: Nosso mapa limpo. Contém 17.089 exames únicos (sem redundância de sobreposição de pacientes). Os targets já estão convertidos em probabilidades prontas para a Loss Function (ex: KL Divergence). Nulos removidos. Contém todos os arquivos que vamos usar.
2. **`train_final.csv`**: Nosso mapa limpo de TREINO com 13671 linhas.
3. **`test_final.csv`**: Nosso mapa limpo de TESTE com 3418 linhas.
4. **Pasta `eeg_processado_mvp/`**: Contém matrizes `.npy` de tamanho fixo ``. Representam exatos 50 segundos de EEG para treino e teste.
5. **Pasta `spectrogram_processado_mvp/`**: Contém matrizes `.npy` de tamanho fixo ``. Representam a janela de 10 minutos para treino e teste.


*O dado original do kaggle que seria o "teste", fica separado com os metadados em test.csv e os .npy em eeg_teste_processado e spectrogram_teste_processado*

#### O que precisa ser feito (de acordo com o que pesquisei):
Para o PyTorch ingerir isso rapidamente sem estourar a memória RAM, você deve criar um `Custom Dataset` que carrega apenas os metadados no `__init__` e busca as matrizes no disco dinamicamente dentro do `__getitem__`.
